# NEWS -> VIEWS: Footage Analysis Pipeline

This notebook runs the footage-first crime analysis pipeline:
1. **Discover** YouTube channels publishing raw BWC footage
2. **Qualify** channels using the 0-100 Channel Qualification Rubric
3. **Score** incidents using the 0-100 Incident Selection Rubric
4. **Build asset inventories** — find all available footage (bodycam, interrogation, court, 911)
5. **Track** pipeline KPIs with weekly dashboards
6. **Sync** everything to Google Sheets

---

### Before you start
Add your API keys to Colab Secrets (key icon in left sidebar):
- `YOUTUBE_API_KEY` (required) — [Get free key](https://console.developers.google.com/)
- `OPENROUTER_API_KEY` (for narrative/case assessment) — [Get key](https://openrouter.ai/)
- `EXA_API_KEY` (for court record search) — [Get key](https://exa.ai/)
- `SHEET_ID` (optional, for Google Sheets sync)

## 1. Setup

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/jj55222/NEWS--VIEWS.git 2>/dev/null || echo 'Repo already cloned'
%cd NEWS--VIEWS
!git checkout claude/footage-analysis-pipeline-Tz2i1
!pip install -q -r requirements.txt

## 2. API Keys

Set your API keys. **Option A** (recommended): Use Colab Secrets sidebar. **Option B**: Set directly below.

In [ ]:
import os

# --- Option A: Colab Secrets (recommended) ---
# Add keys to the Secrets sidebar (key icon on left panel)
try:
    from google.colab import userdata
    os.environ['YOUTUBE_API_KEY'] = userdata.get('YOUTUBE_API_KEY')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    os.environ['EXA_API_KEY'] = userdata.get('EXA_API_KEY')
    print('API keys loaded from Colab Secrets')
except Exception:
    print('Colab Secrets not available. Set keys in Option B below.')

# --- Option B: Set directly (uncomment and fill in) ---
# os.environ['YOUTUBE_API_KEY'] = 'your-key-here'
# os.environ['OPENROUTER_API_KEY'] = 'your-key-here'
# os.environ['EXA_API_KEY'] = 'your-key-here'

# --- Google Sheets ---
os.environ['SHEET_ID'] = '1Id-vc8847w8_vAUF9uQFeOGZU0lgQZjy9PiMFf-y56w'

# Verify
for key in ['YOUTUBE_API_KEY', 'OPENROUTER_API_KEY', 'EXA_API_KEY', 'SHEET_ID']:
    status = 'set' if os.environ.get(key) else 'NOT SET'
    print(f'  {key}: {status}')

## 3. Google Sheets Auth

Upload your service account JSON file (e.g. `yt-api-scrapper-8146742d23f5.json`).
The cell below auto-detects any uploaded `.json` file and renames it for the pipeline.

In [ ]:
import os, shutil
from google.colab import files

print('Upload your service account JSON file:')
uploaded = files.upload()

# Auto-detect: take any uploaded .json file and copy it to service_account.json
for filename in uploaded:
    if filename.endswith('.json'):
        shutil.copy(filename, './service_account.json')
        os.environ['SERVICE_ACCOUNT_PATH'] = './service_account.json'
        print(f'\nDetected: {filename}')
        print(f'Copied to: ./service_account.json')
        print('Service account ready!')
        break
else:
    print('No .json file found in upload. Please upload your service account file.')

## 4. Validate Setup

In [ ]:
# Validate all modules
print('=== Pipeline Ops (Rubric Integrity) ===')
!python pipeline_ops.py

print('\n=== Channel Qualify ===')
!python channel_qualify.py --check

print('\n=== Incident Score ===')
!python incident_score.py --check

print('\n=== KPI Tracker ===')
!python kpi_tracker.py --check

---
## 5. Build Channel Registry

Resolves your 15 videos + 9 explicit channels into the registry (`qualified_channels.json`),
deduplicates, and runs the full 0-100 Channel Qualification Rubric on each.

**Your channels:**
- @JAXSHERIFF (Jacksonville Sheriff) - official PD
- @houstonpolice (Houston Police) - official PD
- @DenverPoliceDept (Denver Police) - official PD
- @AustinPolice (Austin Police) - official PD
- @spdblotter (Seattle Police) - official PD
- UCYa23yHE2e1yJrlYUtyEM2w - use of force reports
- UC2T9FKndXhkgHLk-lKOiCrQ
- UCWu-Puzkp8hv9eSpnWkNxjA
- UCmzeK2lzaBSAQQr2Q0hPArw

Plus channels extracted from your 15 individual videos.

In [ ]:
# Build the channel registry from your seed channels + videos
# Resolves video IDs → channels, resolves @handles, qualifies each
!python channel_qualify.py --seed

## 6. Expand Registry (Optional)

Discover additional BWC channels beyond your seed list via YouTube keyword search
and agency channels from the jurisdiction registry.

In [ ]:
# Optional: expand beyond seed channels with auto-discovery
!python channel_qualify.py --discover

## 7. View Channel Registry

In [ ]:
# List all discovered channels sorted by score
!python channel_qualify.py --list

In [ ]:
# View the full registry as JSON (for detailed inspection)
import json
from pathlib import Path

registry_path = Path('qualified_channels.json')
if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)
    print(f'Total channels: {len(registry)}')
    print(f'Needs review: {sum(1 for v in registry.values() if v.get("needs_manual_review", True))}')
    print()
    # Show top 5 by score
    for entry in sorted(registry.values(), key=lambda x: x.get('score', 0), reverse=True)[:5]:
        print(f"[{entry['score']:2d}] {entry['channel_name']}")
        print(f"     {entry['channel_url']}")
        print(f"     Source tier: {entry['channel_data']['source_tier']}")
        print(f"     Raw footage: {entry['channel_data']['raw_footage_ratio']:.0%}")
        print()
else:
    print('No registry found. Run discovery first (Step 5).')

## 8. Set Watermarks (Manual Review)

After reviewing each channel's videos, set the watermark level.
This adds up to 20 points to the channel's score.

Levels:
- `none` — No channel watermark (20 pts)
- `small` — Small, unobtrusive corner watermark (12 pts)
- `moderate` — Visible but doesn't block footage (5 pts)
- `heavy` — Heavy branding/overlays throughout (0 pts)

In [ ]:
# Set watermark for a specific channel
# Replace CHANNEL_ID and LEVEL with actual values

CHANNEL_ID = 'UCxxxxxx'  # <-- Replace with actual channel ID
WATERMARK_LEVEL = 'none'  # <-- none | small | moderate | heavy

!python channel_qualify.py --set-watermark {CHANNEL_ID} {WATERMARK_LEVEL}

---
## 9. Score Incidents

Score individual videos or batch-score from qualified channels.
Uses the 0-100 Incident Selection Rubric (replaces PASS/KILL).

## 9a. Score Your Specific Videos

Score the individual videos you identified. Each gets a 0-100 incident score.

In [ ]:
# Score all your identified videos
your_videos = [
    "rykYVUNbAs0", "rK3EyLmXPzQ", "Cl_xpyMkOTQ", "SciU3RCrTe0",
    "IZlrbGlbNjM", "-yiwunSIu6U", "c9PUhtIDVC0", "rWyXq4RdNu4",
    "d_hWzbF6XOM", "hNwyKEDNSR0", "fBw_K0iltFI", "AHwOMhDtnDs",
    "4htpzGlCXWw", "RXlRIXFjMCw", "89ckVvmtNNw",
]

for vid in your_videos:
    print(f"\n{'='*60}")
    !python incident_score.py --video {vid}

# Show ranked results
print("\n\n" + "="*60)
print("ALL SCORES RANKED")
print("="*60)
!python incident_score.py --list

In [ ]:
# Score a single video
VIDEO_ID = 'VIDEO_ID_HERE'  # <-- Replace with actual YouTube video ID

!python incident_score.py --video {VIDEO_ID}

In [ ]:
# Batch score recent videos from all qualified channels
# (channels scoring >= 40 on qualification rubric)
!python incident_score.py --batch --limit 10

In [ ]:
# List all scored incidents
!python incident_score.py --list

In [ ]:
# List only strong candidates and above (score >= 55)
!python incident_score.py --list --min-score 55

---
## 10. Asset Inventory

After scoring, build a complete asset inventory for any video.
Searches YouTube, Vimeo, official portals, and 911 archives for all available footage:
bodycam angles, interrogation, court video, surveillance, 911 calls.

Add `--exa` to also search court records, Reddit, and PACER (uses paid Exa credits).

In [ ]:
# Build asset inventory for a single video
VIDEO_ID = 'VIDEO_ID_HERE'  # <-- Replace with video ID from scoring results

!python asset_inventory.py --video {VIDEO_ID}

In [ ]:
# Build inventories for all your scored videos
import json
from pathlib import Path

scores_path = Path('incident_scores.json')
if scores_path.exists():
    with open(scores_path) as f:
        scores = json.load(f)
    # Process videos scoring >= 40 (WATCHLIST and above)
    for vid, data in sorted(scores.items(), key=lambda x: x[1].get('score', 0), reverse=True):
        if data.get('score', 0) >= 40:
            print(f"\n{'='*60}")
            !python asset_inventory.py --video {vid}
else:
    print('No scored incidents found. Run Step 9 first.')

In [ ]:
# View all asset inventories
!python asset_inventory.py --list

## 10. KPI Dashboard

In [ ]:
!python kpi_tracker.py --dashboard

## 11. Sync to Google Sheets

Push all data to your Google Sheet (requires service_account.json and SHEET_ID).

Creates three new tabs:
- **CHANNEL REGISTRY** — All qualified channels with scores
- **INCIDENT SCORES** — All scored incidents ranked by score
- **KPI DASHBOARD** — Weekly KPI tracking

In [ ]:
!python channel_qualify.py --sync
!python incident_score.py --sync
!python kpi_tracker.py --sync

---
## Quick Reference

### Rubric Thresholds

| Rubric | Tier | Score | Action |
|--------|------|-------|--------|
| **Channel** | ELITE | >= 85 | Priority monitoring, daily scan |
| **Channel** | QUALIFIED | >= 60 | Include in pipeline, weekly scan |
| **Channel** | WATCHLIST | >= 40 | Re-evaluate monthly |
| **Incident** | GREENLIGHT | >= 70 | Full enrichment + content production |
| **Incident** | STRONG CANDIDATE | >= 55 | Prioritize for artifact hunting |
| **Incident** | WATCHLIST | >= 40 | Hold, revisit if new artifacts surface |

### CLI Commands

```
python channel_qualify.py --seed                  # Build registry from your channels + videos
python channel_qualify.py --discover              # Find + score more channels
python channel_qualify.py --score CHANNEL_ID      # Score one channel
python channel_qualify.py --set-watermark ID LEVEL # Manual watermark
python channel_qualify.py --list                  # Show registry

python incident_score.py --video VIDEO_ID         # Score one video
python incident_score.py --batch --limit 10       # Batch from channels
python incident_score.py --list --min-score 55    # Show strong candidates+

python asset_inventory.py --video VIDEO_ID       # Build asset inventory
python asset_inventory.py --video VIDEO_ID --exa # Include court/Reddit/PACER
python asset_inventory.py --list                 # List all inventories

python kpi_tracker.py --dashboard                 # KPI dashboard
```